# Aufgabe B: Spannungs-Strom-Messungen an einem DC-SQUID

Dieses Notebook erstellt zwei auswertbare Plots: eine vollständige V-I-Kennlinie mit extrapolierten kritischen Strömen und eine Darstellung der linearen Außenbereiche zur Bestimmung von $R_N/2$.

In [ ]:
# Aufgabe B: V-I-Kennlinie eines DC-SQUIDs
# Publikationsreife Auswertung mit Bestimmung von 2 I_C, R_N/2 und I_C R_N

from pathlib import Path
from matplotlib import rcParams
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.style as mtlstyle

from scipy.optimize import curve_fit

# -----------------------------
# Einstellungen
# -----------------------------
DATA_FILE = Path("10_26_2011-07_21_43_internal_flux_0_nah.csv")

# Nur die äußeren, eindeutig linearen Bereiche werden gefittet.
# Der Wert kann manuell angepasst werden, falls die markierten Fitbereiche im Plot nicht optimal sind.
FIT_MIN_ABS_I = None      # None = automatisch über Quantil bestimmen
FIT_QUANTILE = 0.75       # automatische Wahl: oberste 25 % von |I|

OUT_DIR = Path("plots_B")
OUT_DIR.mkdir(exist_ok=True)

mtlstyle.use("seaborn-v0_8-paper")

rcParams["figure.dpi"] = 300
rcParams["figure.figsize"] = (5, 3)

# -----------------------------
# Daten einlesen
# -----------------------------
df = pd.read_csv(DATA_FILE)
df.columns = df.columns.str.replace('"', '', regex=False).str.strip()

I = df["0_x"].to_numpy(dtype=float)   # Biasstrom in µA
U = df["0_y"].to_numpy(dtype=float)   # Spannung in µV

# Nach Strom sortieren, damit Fits und Linien sauber gezeichnet werden
sort_idx = np.argsort(I)
I = I[sort_idx]
U = U[sort_idx]

# -----------------------------
# Fitfunktionen
# -----------------------------
def linear(x, a, b):
    """Lineares Modell U = a I + b. Wegen µV/µA hat a direkt die Einheit Ohm."""
    return a * x + b

def fit_line(x, y):
    popt, pcov = curve_fit(linear, x, y)
    perr = np.sqrt(np.diag(pcov))
    return popt, perr

# Fitbereich automatisch oder manuell festlegen
if FIT_MIN_ABS_I is None:
    fit_min_abs_I = np.quantile(np.abs(I), FIT_QUANTILE)
else:
    fit_min_abs_I = float(FIT_MIN_ABS_I)

mask_neg = I < -fit_min_abs_I
mask_pos = I >  fit_min_abs_I

if mask_neg.sum() < 3 or mask_pos.sum() < 3:
    raise ValueError("Zu wenige Punkte im linearen Fitbereich. FIT_MIN_ABS_I verkleinern oder Daten prüfen.")

I_neg, U_neg = I[mask_neg], U[mask_neg]
I_pos, U_pos = I[mask_pos], U[mask_pos]

popt_neg, perr_neg = fit_line(I_neg, U_neg)
popt_pos, perr_pos = fit_line(I_pos, U_pos)

a_neg, b_neg = popt_neg
a_pos, b_pos = popt_pos

# x-Achsenschnittpunkte der extrapolierten linearen Äste: U = 0
I0_neg = -b_neg / a_neg
I0_pos = -b_pos / a_pos

# Die Schnittpunkte entsprechen betragsmäßig dem kritischen Gesamtstrom des SQUIDs 2 I_C.
Ic_squid_neg = abs(I0_neg)
Ic_squid_pos = abs(I0_pos)
Ic_squid = np.mean([Ic_squid_neg, Ic_squid_pos])       # 2 I_C
Ic_junction = Ic_squid / 2                             # I_C eines Josephson-Kontakts

# Die Steigung der äußeren linearen Äste entspricht R_N/2.
RN_half_neg = abs(a_neg)
RN_half_pos = abs(a_pos)
RN_half = np.mean([RN_half_neg, RN_half_pos])
RN = 2 * RN_half

# Charakteristische Spannung I_C R_N.
# Numerisch entspricht das auch (2 I_C) * (R_N/2).
char_voltage = Ic_junction * RN

# -----------------------------
# Plot B.1: gesamte Kennlinie mit Fit und kritischem Strom
# -----------------------------
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.scatter(I, U, s=10, alpha=0.75, label="Messdaten")

# Fitbereiche hervorheben
#ax.scatter(I_neg, U_neg, s=16, marker="o", label="Fitbereich")
#ax.scatter(I_pos, U_pos, s=16, marker="o")

# Fitlinien über einen etwas größeren Bereich zeichnen
I_fit_neg = np.linspace(I_neg.min(), min(0, I_neg.max() + 0.2 * abs(I_neg.max())), 300)
I_fit_pos = np.linspace(max(0, I_pos.min() - 0.2 * abs(I_pos.min())), I_pos.max(), 300)
ax.plot(I_fit_neg, linear(I_fit_neg, *popt_neg), label="Linearer Fit", color="orange")
ax.plot(I_fit_pos, linear(I_fit_pos, *popt_pos), color="orange")

# Kritische Ströme markieren
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.axvline(I0_neg, ls="--", lw=1.2, label=r"$\pm 2I_\mathrm{C}$")
ax.axvline(I0_pos, ls="--", lw=1.2)
ax.scatter([I0_neg, I0_pos], [0, 0], marker="x", s=70, zorder=5)

ax.set_title("V-I-Kennlinie des DC-SQUIDs")
ax.set_xlabel(r"Biasstrom $I$ / $\mu$A")
ax.set_ylabel(r"Spannung $U$ / $\mu$V")
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(OUT_DIR / "B1_VI_Kennlinie_mit_Ic.png")
fig.savefig(OUT_DIR / "B1_VI_Kennlinie_mit_Ic.pdf")
plt.show()

# -----------------------------
# Plot B.2: nur Fitbereiche zur Bestimmung von R_N/2
# -----------------------------
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.scatter(I_neg, U_neg, s=18, label="Negativer Ast")
ax.scatter(I_pos, U_pos, s=18, label="Positiver Ast")
ax.plot(I_neg, linear(I_neg, *popt_neg), label=rf"Fit neg.: $R_N/2={RN_half_neg:.2f}\,\Omega$")
ax.plot(I_pos, linear(I_pos, *popt_pos), label=rf"Fit pos.: $R_N/2={RN_half_pos:.2f}\,\Omega$")
ax.set_title(r"Lineare Außenbereiche zur Bestimmung von $R_N/2$")
ax.set_xlabel(r"Biasstrom $I$ / $\mu$A")
ax.set_ylabel(r"Spannung $U$ / $\mu$V")
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(OUT_DIR / "B2_RN_Bestimmung.png")
fig.savefig(OUT_DIR / "B2_RN_Bestimmung.pdf")
plt.show()

# -----------------------------
# Ergebnisse ausgeben
# -----------------------------
print("Auswertung Aufgabe B")
print("-" * 55)
print(f"Datei: {DATA_FILE.name}")
print(f"Fitbereich: |I| > {fit_min_abs_I:.2f} µA")
print()
print("Lineare Fits U = a I + b:")
print(f"negativer Ast: a = {a_neg:.4f} Ω, b = {b_neg:.3f} µV")
print(f"positiver Ast: a = {a_pos:.4f} Ω, b = {b_pos:.3f} µV")
print()
print("Kritischer Strom:")
print(f"Schnittpunkt negativ: {I0_neg:.2f} µA")
print(f"Schnittpunkt positiv: {I0_pos:.2f} µA")
print(f"2 I_C = {Ic_squid:.2f} µA")
print(f"I_C = {Ic_junction:.2f} µA")
print()
print("Normalwiderstand:")
print(f"R_N/2 = {RN_half:.2f} Ω")
print(f"R_N = {RN:.2f} Ω")
print()
print("Charakteristische Spannung:")
print(f"I_C · R_N = {char_voltage:.2f} µV")
print()
print(f"Plots gespeichert in: {OUT_DIR.resolve()}")

